# PISA Chain間 Interface 解析

PDB または mmCIF 構造ファイルを入力として、ローカル PISA により chain 間 interface を解析します。

**入力形式:** `.pdb`, `.cif` / `.mmcif`

**出力 (`results/`):**
- `{PDBID}_chainA-B_pisa_interface.csv` — interface 残基情報（`chain`, `resnum` で RSCC / Q-score 等と merge 可能）
- `{PDBID}_chainA-B_BSA.png` — 残基番号 vs BSA 棒グラフ


## 0. Python 依存パッケージ

初回のみ実行してください。

In [ ]:
from __future__ import annotations

import importlib.util
import subprocess
import sys

REQUIRED = ["pandas", "numpy", "matplotlib", "seaborn", "biopython"]
missing = [pkg for pkg in REQUIRED if importlib.util.find_spec(
    "Bio" if pkg == "biopython" else pkg
) is None]

if missing:
    print(f"Installing: {', '.join(missing)}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("All required Python packages are already installed.")


## 1. ライブラリ読み込み & PISA 実行ファイル確認

必要ライブラリ: pandas, numpy, matplotlib, seaborn, biopython, gemmi（任意）, subprocess, pathlib


In [ ]:
from __future__ import annotations

import importlib.util
import os
import re
import shutil
import subprocess
import sys
import uuid
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from Bio.PDB import MMCIFParser, PDBParser, PPBuilder
from Bio.PDB.MMCIF2Dict import MMCIF2Dict

try:
    import gemmi
    HAS_GEMMI = True
except ImportError:
    HAS_GEMMI = False

# --- PISA 実行ファイル探索 ---
PISA_CANDIDATES = [
    Path("/Applications/ccp4-8.0/bin/pisa"),
    Path("/Applications/ccp4-7.1/bin/pisa"),
    Path("/usr/local/bin/pisa"),
]

def find_pisa_executable() -> Path | None:
    env_path = os.environ.get("PISA_EXE")
    if env_path and Path(env_path).is_file():
        return Path(env_path)
    which = shutil.which("pisa")
    if which:
        return Path(which)
    for candidate in PISA_CANDIDATES:
        if candidate.is_file() and os.access(candidate, os.X_OK):
            return candidate
    return None

PISA_EXE = find_pisa_executable()

if PISA_EXE is None:
    raise FileNotFoundError(
        "PISA 実行ファイルが見つかりません。\n"
        "CCP4 をインストールし、pisa が PATH に含まれることを確認してください。\n"
        "例: /Applications/ccp4-8.0/bin/pisa\n"
        "環境変数 PISA_EXE にフルパスを設定することもできます。"
    )

print(f"PISA executable : {PISA_EXE}")
print(f"gemmi available : {HAS_GEMMI}")

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "results").exists():
    pass
elif (NOTEBOOK_DIR / "structure_tools" / "PISA").exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "structure_tools" / "PISA"
RESULTS_DIR = NOTEBOOK_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Results directory: {RESULTS_DIR}")


## 2. 入力パラメータ

解析対象の構造ファイルと interface chain pair を指定します。


In [ ]:
# ===== ユーザー設定 =====
STRUCTURE_FILE = Path("../RSCC/work/6xez/6xez.pdb")  # .pdb または .cif / .mmcif
CHAIN1 = "A"
CHAIN2 = "B"

# PISA セッション作業ディレクトリ（DATA_ROOT）
PISA_WORK_DIR = NOTEBOOK_DIR / "work" / "pisa_sessions"
PISA_WORK_DIR.mkdir(parents=True, exist_ok=True)

if not STRUCTURE_FILE.exists():
    raise FileNotFoundError(f"構造ファイルが見つかりません: {STRUCTURE_FILE.resolve()}")

print(f"Structure file : {STRUCTURE_FILE.resolve()}")
print(f"Target interface: {CHAIN1} - {CHAIN2}")


## 3. 関数定義

解析処理を以下の関数に整理します。


In [ ]:
from __future__ import annotations

def parse_structure(structure_path: Path) -> dict:
    """PDB/mmCIF からメタ情報を取得する。"""
    suffix = structure_path.suffix.lower()
    meta = {
        "pdb_id": structure_path.stem.upper()[:4],
        "title": None,
        "resolution": None,
        "method": None,
        "assembly": None,
        "structure_path": structure_path,
    }

    if suffix in {".cif", ".mmcif"}:
        cif_dict = MMCIF2Dict(str(structure_path))
        meta["pdb_id"] = (
            cif_dict.get("_entry.id", [meta["pdb_id"]])[0].upper()
        )
        meta["title"] = cif_dict.get("_struct.title", [None])[0]
        meta["method"] = cif_dict.get("_exptl.method", [None])[0]
        for key in ("_refine.ls_d_res_high", "_reflns.d_resolution_high", "_em_3d_reconstruction.resolution"):
            if key in cif_dict:
                try:
                    meta["resolution"] = float(cif_dict[key][0])
                    break
                except (ValueError, IndexError):
                    pass
        if "_pdbx_struct_assembly.id" in cif_dict:
            asm_ids = cif_dict["_pdbx_struct_assembly.id"]
            asm_details = cif_dict.get("_pdbx_struct_assembly.details", asm_ids)
            meta["assembly"] = "; ".join(
                f"{i} ({d})" for i, d in zip(asm_ids, asm_details)
            )
    else:
        with open(structure_path) as fh:
            for line in fh:
                if line.startswith("HEADER"):
                    parts = line.split()
                    if len(parts) >= 5:
                        meta["pdb_id"] = parts[-1].strip()[:4].upper()
                elif line.startswith("TITLE") and meta["title"] is None:
                    meta["title"] = line[10:].strip()
                elif line.startswith("EXPDTA"):
                    meta["method"] = line[10:].strip()
                elif line.startswith("REMARK   2 RESOLUTION."):
                    m = re.search(r"([\d.]+)\s*ANGSTROMS", line)
                    if m:
                        meta["resolution"] = float(m.group(1))
                elif line.startswith("REMARK 350") and meta["assembly"] is None:
                    meta["assembly"] = "see REMARK 350 in PDB file"

    if HAS_GEMMI and meta["title"] is None:
        try:
            st = gemmi.read_structure(str(structure_path))
            if st.name:
                meta["pdb_id"] = st.name.upper()[:4]
        except Exception:
            pass

    return meta


def _fetch_rcsb_chain_descriptions(pdb_id: str) -> dict[str, str]:
    """RCSB REST API から entity 説明を取得する（任意）。"""
    import json as _json
    from urllib.request import urlopen

    base = "https://data.rcsb.org/rest/v1/core"
    mapping: dict[str, str] = {}
    try:
        with urlopen(f"{base}/entry/{pdb_id.upper()}", timeout=30) as resp:
            entry = _json.loads(resp.read().decode())
        container = entry.get("rcsb_entry_container_identifiers", {})

        for entity_id in container.get("polymer_entity_ids", []):
            with urlopen(f"{base}/polymer_entity/{pdb_id.upper()}/{entity_id}", timeout=30) as resp:
                entity = _json.loads(resp.read().decode())
            desc = entity.get("rcsb_polymer_entity", {}).get("pdbx_description", "Unknown")
            strands = entity.get("entity_poly", {}).get("pdbx_strand_id", "")
            for chain_id in [s.strip() for s in strands.replace(" ", "").split(",") if s.strip()]:
                mapping[chain_id] = desc

        for entity_id in container.get("non_polymer_entity_ids", []):
            with urlopen(f"{base}/nonpolymer_entity/{pdb_id.upper()}/{entity_id}", timeout=30) as resp:
                entity = _json.loads(resp.read().decode())
            nonpoly = entity.get("pdbx_entity_nonpoly", {})
            desc = (
                nonpoly.get("name")
                or entity.get("rcsb_nonpolymer_entity", {}).get("pdbx_description")
                or nonpoly.get("comp_id")
                or "Non-polymer entity"
            )
            for chain_id in entity.get("rcsb_nonpolymer_entity_container_identifiers", {}).get("asym_ids", []):
                mapping[chain_id] = desc
    except Exception:
        return mapping
    return mapping


def get_chain_information(structure_path: Path, pdb_id: str | None = None) -> pd.DataFrame:
    """各 chain の molecule 名と残基数を表形式で返す。"""
    suffix = structure_path.suffix.lower()
    if suffix in {".cif", ".mmcif"}:
        parser = MMCIFParser(QUIET=True)
    else:
        parser = PDBParser(QUIET=True)

    structure = parser.get_structure("model", str(structure_path))
    rcsb_names = _fetch_rcsb_chain_descriptions(pdb_id) if pdb_id else {}

    rows = []
    for model in structure:
        for chain in model:
            chain_id = chain.id
            residues = [r for r in chain if r.id[0] == " "]
            res_count = len(residues)

            molecule_name = rcsb_names.get(chain_id)
            if molecule_name is None and HAS_GEMMI and suffix in {".cif", ".mmcif"}:
                try:
                    block = gemmi.cif.read(str(structure_path))[0]
                    for row in block.find("_entity_poly.pdbx_strand_id", ["_entity_poly.pdbx_target_identifier"]):
                        strands = [s.strip() for s in row.str(0).replace(",", " ").split()]
                        if chain_id in strands:
                            entity_id = row.str(1).split()[0] if row.str(1) else None
                            if entity_id:
                                for er in block.find("_entity_name_com.entity_id", ["_entity_name_com.name"]):
                                    if er.str(0) == entity_id:
                                        molecule_name = er.str(1)
                                        break
                except Exception:
                    pass

            if molecule_name is None:
                resnames = {r.get_resname() for r in residues[:20]}
                molecule_name = "/".join(sorted(resnames)) if resnames else "Unknown"

            rows.append({
                "chain_id": chain_id,
                "molecule_name": molecule_name,
                "residue_count": res_count,
            })
        break

    return pd.DataFrame(rows)


def _write_pisa_config(data_root: Path) -> Path:
    """PISA 用設定ファイルを生成する。"""
    cfg_path = data_root / "pisa.cfg"
    if cfg_path.exists():
        return cfg_path

    result = subprocess.run(
        [str(PISA_EXE), "-cfg-template"],
        capture_output=True, text=True, check=True,
    )
    lines = result.stdout.splitlines()
    out = []
    for line in lines:
        if line.strip() == "DATA_ROOT":
            out.append("DATA_ROOT")
            out.append(str(data_root.resolve()))
            continue
        if out and out[-1] == "DATA_ROOT":
            continue
        out.append(line)
    cfg_path.write_text("\n".join(out) + "\n")
    return cfg_path


def run_pisa(structure_path: Path, session_name: str | None = None, work_dir: Path | None = None) -> dict:
    """PISA -analyse を subprocess で実行する。"""
    work_dir = work_dir or (NOTEBOOK_DIR / "work" / "pisa_sessions")
    work_dir.mkdir(parents=True, exist_ok=True)
    cfg_path = _write_pisa_config(work_dir)

    if session_name is None:
        session_name = f"pisa_{structure_path.stem}_{uuid.uuid4().hex[:8]}"

    cmd = [str(PISA_EXE), session_name, "-analyse", str(structure_path.resolve()), str(cfg_path)]
    print("Running:", " ".join(cmd))
    proc = subprocess.run(cmd, capture_output=True, text=True)
    stdout = proc.stdout or ""
    stderr = proc.stderr or ""
    if proc.returncode != 0:
        raise RuntimeError(f"PISA analyse failed (exit {proc.returncode}):\n{stdout}\n{stderr}")
    if "structure read and understood" not in stdout.lower() and "results written" not in stdout.lower():
        print(stdout)
        if stderr:
            print(stderr)

    return {
        "session_name": session_name,
        "cfg_path": cfg_path,
        "work_dir": work_dir,
        "stdout": stdout,
    }


def get_interface_summary(session_name: str, cfg_path: Path) -> pd.DataFrame:
    """PISA -list interfaces の出力を DataFrame に変換する。"""
    cmd = [str(PISA_EXE), session_name, "-list", "interfaces", str(cfg_path)]
    proc = subprocess.run(cmd, capture_output=True, text=True, check=True)
    text = proc.stdout

    rows = []
    pattern = re.compile(
        r"^\s*(\d+)\s+\d+\s*\|\s*(\S+)\s*\|\s*(\S+)\s+.*?\|\s*([\d.]+)\s+([-\d.]+)",
    )
    for line in text.splitlines():
        m = pattern.match(line)
        if not m:
            continue
        rows.append({
            "serial": int(m.group(1)),
            "chain1": m.group(2).strip("[]"),
            "chain2": m.group(3).strip("[]"),
            "interface_area": float(m.group(4)),
            "deltaG": float(m.group(5)),
        })

    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError("PISA interface list が空です。出力を確認してください。")
    return df


def _find_interface_serial(summary: pd.DataFrame, chain1: str, chain2: str) -> int:
    """指定 chain pair に対応する interface serial を返す。"""
    c1, c2 = chain1.strip(), chain2.strip()
    hit = summary[
        ((summary["chain1"] == c1) & (summary["chain2"] == c2))
        | ((summary["chain1"] == c2) & (summary["chain2"] == c1))
    ]
    if hit.empty:
        available = summary[["chain1", "chain2", "interface_area"]].to_string(index=False)
        raise ValueError(
            f"Interface {c1}-{c2} が見つかりません。\n利用可能な interface:\n{available}"
        )
    return int(hit.iloc[0]["serial"])


def _parse_residue_section(lines: list[str], structure_label: str, interface_pair: str) -> list[dict]:
    """PISA detail 出力の Interfacing Residues セクションを解析する。"""
    residue_re = re.compile(
        r"^\s*\d+\s*\|\s*([Is])\s*\|\s*(\w+):(\w+)\s+(\d+)\s*\|\s*([HSDC\s])\s*\|\s*([\d.]+)\s+([\d.]+)\s+([-\d.]+)"
    )
    rows = []
    for line in lines:
        m = residue_re.match(line)
        if not m:
            continue
        rows.append({
            "interface_pair": interface_pair,
            "chain": m.group(2),
            "resname": m.group(3),
            "resnum": int(m.group(4)),
            "Structure1": structure_label,
            "HSDC": m.group(5).strip() or None,
            "ASA": float(m.group(6)),
            "BSA": float(m.group(7)),
            "delta_iG": float(m.group(8)),
        })
    return rows


def extract_interface_residues(
    session_name: str,
    cfg_path: Path,
    chain1: str,
    chain2: str,
    summary: pd.DataFrame | None = None,
) -> pd.DataFrame:
    """指定 interface の残基レポートを取得する。"""
    if summary is None:
        summary = get_interface_summary(session_name, cfg_path)

    serial = _find_interface_serial(summary, chain1, chain2)
    interface_pair = f"{chain1}-{chain2}"

    cmd = [str(PISA_EXE), session_name, "-detail", "interface", str(serial), str(cfg_path)]
    proc = subprocess.run(cmd, capture_output=True, text=True, check=True)
    text = proc.stdout
    lines = text.splitlines()

    rows = []
    section = None
    section_lines = []
    for line in lines:
        if "4. Interfacing Residues: Structure 1" in line:
            if section == "Structure 1" and section_lines:
                rows.extend(_parse_residue_section(section_lines, "Structure 1", interface_pair))
            section = "Structure 1"
            section_lines = []
            continue
        if "5. Interfacing Residues: Structure 2" in line:
            if section == "Structure 1" and section_lines:
                rows.extend(_parse_residue_section(section_lines, "Structure 1", interface_pair))
            section = "Structure 2"
            section_lines = []
            continue
        if section and (line.strip().startswith("Residues with most") or line.strip().startswith("----'")):
            rows.extend(_parse_residue_section(section_lines, section, interface_pair))
            section = None
            section_lines = []
            continue
        if section:
            section_lines.append(line)

    if section and section_lines:
        rows.extend(_parse_residue_section(section_lines, section, interface_pair))

    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError(f"Interface {interface_pair} (serial={serial}) の残基情報を取得できませんでした。")

    # merge 用キー: interface_pair, chain, resnum
    df = df.sort_values(["chain", "resnum"]).reset_index(drop=True)
    return df


def export_interface_csv(df: pd.DataFrame, pdb_id: str, chain1: str, chain2: str, out_dir: Path) -> Path:
    """interface 残基 CSV を出力する。"""
    out_dir.mkdir(parents=True, exist_ok=True)
    filename = f"{pdb_id.upper()}_chain{chain1}-{chain2}_pisa_interface.csv"
    out_path = out_dir / filename

    columns = [
        "interface_pair", "chain", "resname", "resnum",
        "Structure1", "HSDC", "ASA", "BSA", "delta_iG",
    ]
    df[columns].to_csv(out_path, index=False)
    print(f"CSV saved: {out_path}")
    return out_path


def plot_bsa_barplot(
    df: pd.DataFrame,
    pdb_id: str,
    chain1: str,
    chain2: str,
    out_dir: Path,
    dpi: int = 300,
) -> tuple[plt.Figure, Path]:
    """残基番号 vs BSA の棒グラフを作成・保存する。"""
    plot_df = df[df["BSA"] > 0].copy()
    if plot_df.empty:
        plot_df = df.copy()

    chain_colors = {chain1: "#1f77b4", chain2: "#d62728"}
    default_color = "#888888"

    fig, ax = plt.subplots(figsize=(14, 5))
    for chain_id, sub in plot_df.groupby("chain"):
        color = chain_colors.get(chain_id, default_color)
        ax.bar(sub["resnum"], sub["BSA"], width=1.0, color=color, label=f"Chain {chain_id}", alpha=0.85)

    ax.set_xlabel("Residue Number")
    ax.set_ylabel("BSA (Å²)")
    ax.set_title(f"{pdb_id.upper()} Chain {chain1}-{chain2} Interface")
    ax.legend()
    sns.despine(ax=ax)
    fig.tight_layout()

    out_dir.mkdir(parents=True, exist_ok=True)
    png_path = out_dir / f"{pdb_id.upper()}_chain{chain1}-{chain2}_BSA.png"
    fig.savefig(png_path, dpi=dpi, bbox_inches="tight")
    print(f"Plot saved: {png_path}")
    return fig, png_path


def main(
    structure_path: Path,
    chain1: str,
    chain2: str,
    results_dir: Path | None = None,
) -> dict:
    """PISA interface 解析のメインワークフロー。"""
    results_dir = results_dir or RESULTS_DIR

    meta = parse_structure(structure_path)
    chain_df = get_chain_information(structure_path, meta["pdb_id"])

    print("=" * 50)
    print(f"PDB ID     : {meta['pdb_id']}")
    if meta["title"]:
        print(f"Title      : {meta['title']}")
    if meta["resolution"] is not None:
        print(f"Resolution : {meta['resolution']:.2f} Å")
    if meta["method"]:
        print(f"Method     : {meta['method']}")
    if meta["assembly"]:
        print(f"Assembly   : {meta['assembly']}")
    print("=" * 50)
    print(chain_df.to_string(index=False))

    pisa_info = run_pisa(structure_path)
    summary = get_interface_summary(pisa_info["session_name"], pisa_info["cfg_path"])
    print("\n--- All interfaces ---")
    print(summary[["chain1", "chain2", "interface_area", "deltaG"]].to_string(index=False))

    selected = summary[
        ((summary["chain1"] == chain1) & (summary["chain2"] == chain2))
        | ((summary["chain1"] == chain2) & (summary["chain2"] == chain1))
    ]
    print(f"\n--- Selected interface: {chain1} - {chain2} ---")
    print(selected[["chain1", "chain2", "interface_area", "deltaG"]].to_string(index=False))

    residues = extract_interface_residues(
        pisa_info["session_name"], pisa_info["cfg_path"], chain1, chain2, summary
    )
    csv_path = export_interface_csv(residues, meta["pdb_id"], chain1, chain2, results_dir)
    fig, png_path = plot_bsa_barplot(residues, meta["pdb_id"], chain1, chain2, results_dir)

    return {
        "meta": meta,
        "chain_df": chain_df,
        "summary": summary,
        "residues": residues,
        "csv_path": csv_path,
        "png_path": png_path,
        "pisa_info": pisa_info,
    }


## 4. 構造ファイルのメタ情報取得

`parse_structure()` と `get_chain_information()` を実行します。


In [ ]:
meta = parse_structure(STRUCTURE_FILE)
chain_df = get_chain_information(STRUCTURE_FILE, meta["pdb_id"])

print(f"PDB ID     : {meta['pdb_id']}")
if meta['title']:
    print(f"Title      : {meta['title']}")
if meta['resolution'] is not None:
    print(f"Resolution : {meta['resolution']:.2f} Å")
if meta['method']:
    print(f"Method     : {meta['method']}")
if meta['assembly']:
    print(f"Assembly   : {meta['assembly']}")

chain_df


## 5. PISA 実行 & Interface 一覧

`run_pisa()` → `get_interface_summary()` で全 interface を DataFrame 表示します。


In [ ]:
pisa_info = run_pisa(STRUCTURE_FILE)
interface_summary = get_interface_summary(pisa_info["session_name"], pisa_info["cfg_path"])
interface_summary[["chain1", "chain2", "interface_area", "deltaG"]]


## 6. Interface 選択

`CHAIN1`, `CHAIN2` で指定した chain pair の interface を抽出します。


In [ ]:
selected_interface = interface_summary[
    ((interface_summary["chain1"] == CHAIN1) & (interface_summary["chain2"] == CHAIN2))
    | ((interface_summary["chain1"] == CHAIN2) & (interface_summary["chain2"] == CHAIN1))
]
selected_interface[["chain1", "chain2", "interface_area", "deltaG"]]


## 7. Interface 残基情報取得

PISA residue interface report を解析し、以下の列を取得します。

| 列 | 説明 |
|----|------|
| `interface_pair` | 例: `A-B` |
| `chain` | chain ID（merge キー） |
| `resname` | 残基名 |
| `resnum` | 残基番号（merge キー） |
| `Structure1` | Structure 1 / Structure 2 |
| `HSDC` | 二次構造 (H/S/D/C) |
| `ASA` | 表面積 (Å²) |
| `BSA` | 埋没表面積 (Å²) |
| `delta_iG` | 残基ごとの solvation energy 変化 |


In [ ]:
interface_residues = extract_interface_residues(
    pisa_info["session_name"],
    pisa_info["cfg_path"],
    CHAIN1,
    CHAIN2,
    interface_summary,
)
interface_residues.head(10)


## 8. CSV 出力

`results/{PDBID}_chainA-B_pisa_interface.csv` に保存します。

> **将来拡張:** `chain`, `resnum`, `interface_pair` を必須キーとして保存し、RSCC / Q-score / Ramachandran 等と `pandas.merge` しやすい形式にしています。


In [ ]:
csv_path = export_interface_csv(
    interface_residues,
    meta["pdb_id"],
    CHAIN1,
    CHAIN2,
    RESULTS_DIR,
)
csv_path


## 9. BSA 可視化 & 画像保存

残基番号 vs BSA の棒グラフ（Chain A: 青, Chain B: 赤）を `results/{PDBID}_chainA-B_BSA.png` (dpi=300) に保存します。


In [ ]:
fig, png_path = plot_bsa_barplot(
    interface_residues,
    meta["pdb_id"],
    CHAIN1,
    CHAIN2,
    RESULTS_DIR,
    dpi=300,
)
plt.show()
png_path


## 10. main() 一括実行（オプション）

上記セクションを個別に実行する代わりに、`main()` で一括実行することもできます。


In [ ]:
# result = main(STRUCTURE_FILE, CHAIN1, CHAIN2, RESULTS_DIR)


## 11. 解析結果の確認

CSV を再読込し、`head()` と `describe()` で結果を確認します。


In [ ]:
result_csv = pd.read_csv(csv_path)

print("=== head() ===")
display(result_csv.head())

print("\n=== describe() ===")
display(result_csv.describe(include="all"))

print(f"\n出力ファイル:")
print(f"  CSV : {csv_path}")
print(f"  PNG : {png_path}")
